# Lab 12: LSTM Model for Time-Series Forecasting

**Name:** Zubair Moeen  
**Reg Number:** 22jzele0463  
**Lab:** Machine Learning Lab  
**Lab Supervisor:** Engr.Irshad Ullah  
**University:** UET Peshawar - Campus Nowshera   

## Lab Overview
This notebook develops an LSTM-based deep learning model for time-series forecasting. The code imports forecasting utilities, defines an LSTM architecture, configures callbacks and checkpoints, loads train/validation/test datasets, trains the model, and evaluates forecasting performance using regression metrics.

## Learning Objectives
- Set the working directory and import Scikit-learn, TensorFlow/Keras, and custom time-series utilities.
- Define an LSTM neural network architecture for sequential forecasting data.
- Configure optimizer, model checkpoints, and training monitor callbacks.
- Load training, validation, and testing CSV files for time-series prediction.
- Evaluate LSTM forecasting output using MAE, MSE, RMSE, R2 score, and explained variance.


## Section 1: Working Directory and Library Imports
This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.


In [1]:
import os
os.chdir('Z:\\University\\8th Semester\\ML Lab\\Lab 12')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## Section 2: Model Parameters and LSTM Architecture
The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.


In [5]:
model1 = create_lstm()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 8)          │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20)             │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,301 (12.89 KB)

 Trainable params: 3,301 (12.89 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'Z:\University\8th Semester\ML Lab\Lab 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## Section 3: Checkpoints, Callbacks, and Training Configuration
This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.


In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [10]:
path_dataset = r'Z:\University\8th Semester\ML Lab\Lab 12'

path_tr = os.path.join(path_dataset, 'AEP_train.csv')
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
path_te = os.path.join(path_dataset, 'AEP_test.csv')
path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')

for path in [path_tr, path_v, path_te, path_hourly]:
    print(path, "=>", os.path.exists(path))

df_tr = pd.read_csv(path_tr)
train_set = df_tr.values

df_v = pd.read_csv(path_v)
validation_set = df_v.values

df_te = pd.read_csv(path_te)
test_set = df_te.values

df_hourly = pd.read_csv(path_hourly)

train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape

Z:\University\8th Semester\ML Lab\Lab 12\AEP_train.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_validation.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_test.csv => True
Z:\University\8th Semester\ML Lab\Lab 12\AEP_hourly.csv => True


((84907, 21), (24259, 21), (12130, 21), (121273, 2))

In [11]:
time_steps=24
num_features=21

In [12]:
start = time.time()

train_X, train_y = univariate_multi_step(
    train_set,
    time_steps,
    target_col=0,
    target_len=1
)

validation_X, validation_y = univariate_multi_step(
    validation_set,
    time_steps,
    target_col=0,
    target_len=1
)

test_X, test_y = univariate_multi_step(
    test_set,
    time_steps,
    target_col=0,
    target_len=1
)

print('Time Consumed', time.time() - start, "sec")

Time Consumed 0.4608793258666992 sec


## Section 4: Dataset Loading and Forecast Preparation
The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.


In [13]:
epochs = 10
verbose = 1
batch_size = 32

History = model.fit(
    train_X,
    train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=verbose
)

Epoch 1/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0598 - mae: 0.0598 - mape: 26.6617
Epoch 1: val_loss improved from None to 0.02240, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0403 - mae: 0.0403 - mape: 64.2715 - val_loss: 0.0224 - val_mae: 0.0224 - val_mape: 9.8677
Epoch 2/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0207 - mae: 0.0207 - mape: 37.6049
Epoch 2: val_loss improved from 0.02240 to 0.01398, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0002-loss0.01.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0002-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - loss: 0.0186 - mae: 0.0186 - mape: 368.7582 - val_loss: 0.0140 - val_mae: 0.0140 - val_mape: 6.7315
Epoch 3/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0136 - mae: 0.0136 - mape: 462.6050
Epoch 3: val_loss improved from 0.01398 to 0.01146, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0003-loss0.01.h5



Epoch 3: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0003-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - loss: 0.0128 - mae: 0.0128 - mape: 414.4326 - val_loss: 0.0115 - val_mae: 0.0115 - val_mape: 5.9238
Epoch 4/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0117 - mae: 0.0117 - mape: 93.3588
Epoch 4: val_loss improved from 0.01146 to 0.00987, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0004-loss0.01.h5



Epoch 4: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0004-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - loss: 0.0115 - mae: 0.0115 - mape: 196.1958 - val_loss: 0.0099 - val_mae: 0.0099 - val_mape: 4.6063
Epoch 5/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0109 - mae: 0.0109 - mape: 4.5800
Epoch 5: val_loss did not improve from 0.00987
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - loss: 0.0107 - mae: 0.0107 - mape: 131.9750 - val_loss: 0.0099 - val_mae: 0.0099 - val_mape: 4.4065
Epoch 6/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0100 - mae: 0.0100 - mape: 19.5412
Epoch 6: val_loss improved from 0.00987 to 0.00905, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0006-loss0.01.h5



Epoch 6: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0006-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 32s 12ms/step - loss: 0.0098 - mae: 0.0098 - mape: 54.2758 - val_loss: 0.0090 - val_mae: 0.0090 - val_mape: 4.1336
Epoch 7/10
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0094 - mae: 0.0094 - mape: 78.1085
Epoch 7: val_loss improved from 0.00905 to 0.00861, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0007-loss0.01.h5



Epoch 7: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0007-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - loss: 0.0093 - mae: 0.0093 - mape: 281.9313 - val_loss: 0.0086 - val_mae: 0.0086 - val_mape: 3.8236
Epoch 8/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0089 - mae: 0.0089 - mape: 155.7397
Epoch 8: val_loss improved from 0.00861 to 0.00789, saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5



Epoch 8: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 32s 12ms/step - loss: 0.0089 - mae: 0.0089 - mape: 133.9371 - val_loss: 0.0079 - val_mae: 0.0079 - val_mape: 3.5297
Epoch 9/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0087 - mae: 0.0087 - mape: 429.8537
Epoch 9: val_loss did not improve from 0.00789
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - loss: 0.0086 - mae: 0.0086 - mape: 221.4505 - val_loss: 0.0081 - val_mae: 0.0081 - val_mape: 3.8071
Epoch 10/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0085 - mae: 0.0085 - mape: 267.0731
Epoch 10: val_loss did not improve from 0.00789
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 28s 11ms/step - loss: 0.0085 - mae: 0.0085 - mape: 195.9102 - val_loss: 0.0084 - val_mae: 0.0084 - val_mape: 3.7909


In [14]:
import os
import pickle

path_dataset = r'Z:\University\8th Semester\ML Lab\Lab 12'
path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')

print(os.path.exists(path_scaler))

with open(path_scaler, 'rb') as f:
    scaler = pickle.load(f)

True


c:\Users\engin\.conda\envs\ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
model_path = r'Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5'

model = load_model(model_path, compile=False)

y_pred_scaled = model.predict(test_X)

print("y_pred_scaled.shape =", y_pred_scaled.shape)
print("test_y.shape =", test_y.shape)

def inverse_target(scaled_values, scaler, target_col=0, num_features=21):
    scaled_values = scaled_values.reshape(-1)

    dummy = np.zeros((scaled_values.shape[0], num_features))
    dummy[:, target_col] = scaled_values

    unscaled = scaler.inverse_transform(dummy)

    return unscaled[:, target_col].reshape(-1, 1)

y_pred = inverse_target(
    y_pred_scaled,
    scaler,
    target_col=0,
    num_features=21
)

y_test_unscaled = inverse_target(
    test_y,
    scaler,
    target_col=0,
    num_features=21
)

# Mean Absolute Error (MAE)
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.mean(np.square(y_pred - y_test_unscaled))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(MSE)
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape = ', y_test_unscaled.shape)
print('y_pred.shape = ', y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
y_pred_scaled.shape = (12105, 1)
test_y.shape = (12105, 1)
Mean Absolute Error (MAE): 124.27
Median Absolute Error (MedAE): 99.27
Mean Squared Error (MSE): 26865.33
Root Mean Squared Error (RMSE): 163.91
Mean Absolute Percentage Error (MAPE): 0.86 %
Median Absolute Percentage Error (MDAPE): 0.68 %


y_test_unscaled.shape =  (12105, 1)
y_pred.shape =  (12105, 1)


In [16]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5'
start_epoch= 34

## Section 5: Prediction and Model Evaluation
The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.


In [18]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_load = r'Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_load))
    model = load_model(model_load, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )


    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading Z:\University\8th Semester\ML Lab\Lab 12\\E1-cp-0008-loss0.01.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [19]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0077 - mae: 0.0077 - mape: 230.2263
Epoch 1: val_loss improved from None to 0.00784, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0001-loss0.01.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0001-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - loss: 0.0077 - mae: 0.0077 - mape: 220.2930 - val_loss: 0.0078 - val_mae: 0.0078 - val_mape: 3.4288
Epoch 2/10
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0077 - mae: 0.0077 - mape: 65.9435
Epoch 2: val_loss improved from 0.00784 to 0.00756, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0002-loss0.01.h5



Epoch 2: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0002-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - loss: 0.0077 - mae: 0.0077 - mape: 142.6224 - val_loss: 0.0076 - val_mae: 0.0076 - val_mape: 3.3778
Epoch 3/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0076 - mae: 0.0076 - mape: 61.8587
Epoch 3: val_loss did not improve from 0.00756
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 33s 12ms/step - loss: 0.0076 - mae: 0.0076 - mape: 162.7037 - val_loss: 0.0078 - val_mae: 0.0078 - val_mape: 3.4941
Epoch 4/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0075 - mae: 0.0075 - mape: 2.6739
Epoch 4: val_loss improved from 0.00756 to 0.00756, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0004-loss0.01.h5



Epoch 4: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0004-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - loss: 0.0076 - mae: 0.0076 - mape: 118.1852 - val_loss: 0.0076 - val_mae: 0.0076 - val_mape: 3.3607
Epoch 5/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0075 - mae: 0.0075 - mape: 311.6636
Epoch 5: val_loss did not improve from 0.00756
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - loss: 0.0075 - mae: 0.0075 - mape: 158.4479 - val_loss: 0.0076 - val_mae: 0.0076 - val_mape: 3.3956
Epoch 6/10
2649/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0076 - mae: 0.0076 - mape: 108.8909
Epoch 6: val_loss improved from 0.00756 to 0.00749, saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0006-loss0.01.h5



Epoch 6: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0006-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 31s 12ms/step - loss: 0.0075 - mae: 0.0075 - mape: 100.7618 - val_loss: 0.0075 - val_mae: 0.0075 - val_mape: 3.3293
Epoch 7/10
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0075 - mae: 0.0075 - mape: 16.6453
Epoch 7: val_loss did not improve from 0.00749
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - loss: 0.0075 - mae: 0.0075 - mape: 154.2828 - val_loss: 0.0076 - val_mae: 0.0076 - val_mape: 3.3827
Epoch 8/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0075 - mae: 0.0075 - mape: 8.7339
Epoch 8: val_loss did not improve from 0.00749
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - loss: 0.0074 - mae: 0.0074 - mape: 137.4383 - val_loss: 0.0075 - val_mae: 0.0075 - val_mape: 3.3402
Epoch 9/10
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0074 - mae: 0.0074 - mape: 600.1637
Epoch 9: val_loss did not improve from 0.00749
2653/2653 ━━━━━━━━━


Epoch 10: finished saving model to Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0010-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 28s 11ms/step - loss: 0.0074 - mae: 0.0074 - mape: 89.6044 - val_loss: 0.0074 - val_mae: 0.0074 - val_mape: 3.3342


In [20]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 12\E2-cp-0010-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Mean Absolute Error (MAE): 118.66
Median Absolute Error (MedAE): 92.58
Mean Squared Error (MSE): 24911.03
Root Mean Squared Error (RMSE): 157.83
Mean Absolute Percentage Error (MAPE): 0.82 %
Median Absolute Percentage Error (MDAPE): 0.64 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Final Conclusion
In this lab, an LSTM neural network was implemented for time-series forecasting. The notebook demonstrates sequence-model architecture, callback-based training, data preparation, and regression-based model evaluation.
